In [0]:

CATALOG = "venergy"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

TABELA_BRONZE = f"{CATALOG}.{BRONZE_SCHEMA}.inspecoes"
TABELA_SILVER = f"{CATALOG}.{SILVER_SCHEMA}.inspecoes"

CONFIRMAR_GRAVACAO = False

print("Pipeline Bronze -> Silver: Inspecoes")
print(f"Origem: {TABELA_BRONZE}")
print(f"Destino: {TABELA_SILVER}")
print(f"Gravacao habilitada: {CONFIRMAR_GRAVACAO}")


In [0]:

if not spark.catalog.tableExists(TABELA_BRONZE):
    raise ValueError(f"Tabela Bronze nao encontrada: {TABELA_BRONZE}")

df_bronze = spark.table(TABELA_BRONZE)

print(f"Total de registros na Bronze: {df_bronze.count()}")

df_bronze.printSchema()

display(df_bronze.limit(10))


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType

COLUNAS_OBRIGATORIAS = [
    "id_inspecao",
    "id_uc",
    "data",
    "inspetor",
    "resultado",
    "tipo_irregularidade",
    "kwh_recuperado"
]

colunas_ausentes = [
    coluna for coluna in COLUNAS_OBRIGATORIAS
    if coluna not in df_bronze.columns
]

if colunas_ausentes:
    raise ValueError(
        f"Colunas obrigatorias ausentes: {colunas_ausentes}"
    )

df_tratado = (
    df_bronze
    .withColumn("id_inspecao", F.trim(F.col("id_inspecao").cast("string")))
    .withColumn("id_uc", F.trim(F.col("id_uc").cast("string")))
    .withColumn("inspetor", F.upper(F.trim(F.col("inspetor").cast("string"))))
    .withColumn("resultado", F.upper(F.trim(F.col("resultado").cast("string"))))
    .withColumn(
        "tipo_irregularidade",
        F.upper(F.trim(F.col("tipo_irregularidade").cast("string")))
    )
)

print("Padronizacao inicial configurada.")


In [0]:

df_tratado = df_tratado.withColumn(
    "data",
    F.coalesce(
        F.to_date(F.expr("try_to_timestamp(trim(data), 'yyyy-MM-dd')")),
        F.to_date(F.expr("try_to_timestamp(trim(data), 'dd/MM/yyyy')")),
        F.to_date(F.expr("try_to_timestamp(trim(data), 'dd.MM.yyyy')")),
        F.to_date(F.expr("try_to_timestamp(trim(data), 'MM-dd-yyyy')"))
    )
)

print("Tratamento de datas configurado.")


In [0]:

df_tratado = (
    df_tratado
    .withColumn(
        "resultado",
        F.when(
            F.col("resultado").isin(
                "CONFIRMADA",
                "IRREGULARIDADE CONFIRMADA"
            ),
            F.lit("IRREGULARIDADE CONFIRMADA")
        )
        .when(
            F.col("resultado").isin(
                "NADA CONSTA",
                "SEM IRREGULARIDADE"
            ),
            F.lit("SEM IRREGULARIDADE")
        )
        .otherwise(F.col("resultado"))
    )
    .withColumn(
        "tipo_irregularidade",
        F.when(
            F.col("tipo_irregularidade").isin(
                "", "-", "NULL", "N/A"
            ) | F.col("tipo_irregularidade").isNull(),
            F.lit(None).cast("string")
        )
        .otherwise(F.col("tipo_irregularidade"))
    )
    .withColumn(
        "kwh_recuperado",
        F.regexp_replace(
            F.trim(F.col("kwh_recuperado").cast("string")),
            ",",
            "."
        ).cast(DecimalType(10, 2))
    )
)

print("Normalizacao de resultados, irregularidades e kWh configurada.")


In [0]:

ids_invalidos_bronze = df_tratado.filter(
    F.col("id_inspecao").isNull() |
    (F.col("id_inspecao") == "")
).count()

if ids_invalidos_bronze > 0:
    raise ValueError(
        f"Encontrados {ids_invalidos_bronze} registros com id_inspecao invalido na Bronze."
    )

if "ingestion_datetime" not in df_tratado.columns:
    raise ValueError("Coluna ingestion_datetime nao encontrada na Bronze.")

df_tratado = df_tratado.withColumn(
    "ingestion_datetime",
    F.col("ingestion_datetime").cast("timestamp")
)

colunas_desempate = [
    F.col("ingestion_datetime").desc_nulls_last()
]

for coluna in sorted(df_tratado.columns):
    if coluna not in ("id_inspecao", "ingestion_datetime"):
        colunas_desempate.append(
            F.col(coluna).cast("string").asc_nulls_last()
        )

janela_inspecoes = Window.partitionBy("id_inspecao").orderBy(
    *colunas_desempate
)

df_silver = (
    df_tratado
    .withColumn(
        "numero_linha",
        F.row_number().over(janela_inspecoes)
    )
    .filter(F.col("numero_linha") == 1)
    .drop("numero_linha")
)

print("Deduplicacao por id_inspecao configurada.")


In [0]:

resultados_validos = [
    "IRREGULARIDADE CONFIRMADA",
    "SEM IRREGULARIDADE"
]

df_validacao = df_silver.agg(
    F.count("*").alias("total_registros"),
    F.sum(
        F.when(
            F.col("id_inspecao").isNull() |
            (F.col("id_inspecao") == ""),
            1
        ).otherwise(0)
    ).alias("ids_invalidos"),
    F.sum(
        F.when(
            F.col("id_uc").isNull() |
            (F.col("id_uc") == ""),
            1
        ).otherwise(0)
    ).alias("ucs_invalidas"),
    F.sum(
        F.when(F.col("data").isNull(), 1).otherwise(0)
    ).alias("datas_invalidas"),
    F.sum(
        F.when(
            F.col("resultado").isNull() |
            (~F.col("resultado").isin(resultados_validos)),
            1
        ).otherwise(0)
    ).alias("resultados_invalidos"),
    F.sum(
        F.when(
            F.col("kwh_recuperado").isNull() |
            (F.col("kwh_recuperado") < 0),
            1
        ).otherwise(0)
    ).alias("kwh_invalidos"),
    F.countDistinct("id_inspecao").alias("ids_distintos")
)

validacao = df_validacao.first().asDict()

print(validacao)

campos_invalidos = [
    "ids_invalidos",
    "ucs_invalidas",
    "datas_invalidas",
    "resultados_invalidos",
    "kwh_invalidos"
]

erros = [
    campo for campo in campos_invalidos
    if validacao[campo] != 0
]

if validacao["total_registros"] == 0:
    raise ValueError("A Silver de inspecoes esta vazia.")

if validacao["total_registros"] != validacao["ids_distintos"]:
    raise ValueError("Foram encontrados IDs de inspecao duplicados.")

if erros:
    raise ValueError(f"Falha na qualidade dos dados: {erros}")

print("Validacoes de qualidade concluidas com sucesso.")


In [0]:

if CONFIRMAR_GRAVACAO:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

    (
        df_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TABELA_SILVER)
    )

    print(f"Tabela Silver gravada com sucesso: {TABELA_SILVER}")
else:
    print("Gravacao desabilitada por seguranca.")
    print(f"Tabela de destino: {TABELA_SILVER}")
